In [37]:
#本地训练word2vec_Skip-Gram模型
from gensim.models import Word2Vec
from gensim.models.keyedvectors import KeyedVectors
import pandas as pd

# 假设S是一个分词后的句子列表，Dc和Df是特征词的集合
df = pd.read_csv(r'E:\UAV\中文停用词表分词结果\cut_UAV_data2.csv', encoding='utf-8')

# 确保评论内容列中的所有数据都是字符串类型，并去除NaN值
df['评论内容'] = df['评论内容'].astype(str).dropna()

# 将评论数据转换为句子列表，每个句子是一个单词列表
S = df['评论内容'].apply(lambda x: x.split()).tolist()

Dc = {"电池", "避障", "遥控器", "充电", "信号", "图传", "镜头", "遥控", "云台", "机身", "相机", "摄像头", "4g",
      "桨叶", "摇杆", "固件", "手柄", "电机", "飞控", "gps", "屏幕", "传感器", "软件", "系统", "充电器", "o4", "o3",
      "哈苏", "主摄", "内存卡", "电芯", "螺旋桨", "圈圈", "电线", "rc", "rc2", "硬件", "风扇", "光圈", "散热", "机臂",
      "雷达", "核心", "导航", "激光雷达", "稳定器", "芯片", "模组", "接口", "按钮", "轴承", "显示器", "排线", "叶片",
      "主板", "机翼", "旋翼"}
Df = {"失联", "掉下来", "故障", "下降", "没电", "失控", "乱飞", "坠机", "噪音", "停桨", "破损", "bug", "遮挡","生锈",
      "起雾", "损坏", "抖动", "迫降", "异常", "瑕疵", "降低", "避障关", "缺陷", "发热", "异响", "撞击", "失效", "过热",
      "鼓包", "落地", "关不了", "有雾", "没避障", "卡顿", "延迟", "坠落", "报警", "摔坏", "错误", "太差", "黑屏","划痕"
      "断联", "晃动", "模糊", "进水", "中断", "翻滚", "关机", "断电", "撞机", "侧飞", "降到", "俯冲", "警告", "爆炸"}

# 训练Word2Vec模型（使用skip-gram模型）
model = Word2Vec(S, vector_size=300, window=10, min_count=3, sg=1, workers=4)

# 获取模型的词向量
word_vectors = model.wv

# 函数：计算并打印与给定词相似度最高的10个词
def get_most_similar(vocab_set, top_n=10):
    data = []
    for word in vocab_set:
        if word in word_vectors:
            similar_words = word_vectors.most_similar(word, topn=top_n)
            data.extend([[word, similar_word, similarity] for similar_word, similarity in similar_words])
        else:
            data.append([word, "not in vocabulary", None])
    return pd.DataFrame(data, columns=['Target Word', 'Similar Word', 'Similarity'])

get_most_similar(Dc).to_csv(r'E:\UAV\中文停用词表分词结果\组件相似词1.csv', encoding='utf_8_sig', index=False)
get_most_similar(Df).to_csv(r'E:\UAV\中文停用词表分词结果\故障相似词1.csv', encoding='utf_8_sig', index=False)

print(get_most_similar(Dc))
print(get_most_similar(Df))

    Target Word Similar Word  Similarity
0            相机         CMOS    0.869070
1            相机           露营    0.861989
2            相机         Vlog    0.856977
3            相机         4k60    0.856511
4            相机         dlog    0.853574
..          ...          ...         ...
565          接口           背囊    0.932634
566          接口            插    0.931866
567          接口            灯    0.930912
568          接口           剩余    0.929784
569          接口            闪    0.929568

[570 rows x 3 columns]
    Target Word Similar Word  Similarity
0            错误          工程师    0.963501
1            错误           程序    0.961114
2            错误           找回    0.958054
3            错误           事故    0.953985
4            错误           撞击    0.949394
..          ...          ...         ...
536          过热           剩余    0.955485
537          过热           电压    0.954237
538          过热           接口    0.949462
539          过热           放电    0.945916
540          过热           断电    0

In [39]:
# 1
# 相似度计算结果中挑选出来的要加入Dc和Df的词
selected_for_Dc = set(["罩", "螺旋桨", "螺丝柱", "机体", "壳", "CMOS", "算法", "电源", "整机", "摇杆", "轴臂", "usb", "模块", "天线",
                       "三轴", "外壳", "卡扣", "硬件", "指示灯"])  
selected_for_Df = set(["变形", "磨损", "异常", "停转", "抗风差", "崩溃", "过热", "接触不良", "坏点", "放电","脱落", "丢失", "漂"
                       "干扰", "弱", "歪", "过载", "飘", "断裂", "松动", "报错", "坠毁", "中断", "黑屏", "拉胯", "停不下来", "砸"
                       "撞坏", "回不来", "关机", "不准", "关闭", "关避障", "出错", "误报", "裂开", "烫", "烫手", "偏移", "断开连接", 
                       "掉落", "烧", "延时", "倾斜", "短路"])  

# 将选中的词添加到对应的词典中
Dc = Dc.union(selected_for_Dc)
Df = Df.union(selected_for_Df)

# 打印更新后的词典
print("更新后的Dc:", Dc)
print("更新后的Df:", Df)

# 创建噪声词词典Dn，包含所有不在Dc和Df中的词
# 注意：这里假设S_flat包含了所有分词后的数据集词汇
Dn = set([word for sentence in S for word in sentence]) - Dc - Df

print("噪声词典Dn:", Dn)

更新后的Dc: {'相机', '哈苏', '散热', '云台', '软件', '稳定器', '圈圈', '螺丝柱', '系统', '机体', '模组', '机臂', '轴承', '指示灯', '电芯', '电源', '芯片', 'usb', '叶片', '遥控', '4g', '罩', '硬件', '排线', '遥控器', '桨叶', '传感器', 'rc2', '模块', '显示器', '飞控', '图传', 'o4', '电线', '按钮', '充电器', '机身', '卡扣', 'o3', '镜头', '机翼', '摇杆', '整机', 'CMOS', '主摄', '外壳', '摄像头', '螺旋桨', '算法', '避障', '电池', '风扇', '导航', '主板', '手柄', '雷达', '电机', 'gps', '内存卡', '天线', '固件', '信号', '壳', '光圈', '旋翼', '充电', '核心', '屏幕', '三轴', '激光雷达', 'rc', '轴臂', '接口'}
更新后的Df: {'砸撞坏', '警告', '爆炸', '翻滚', '掉落', '坠落', '松动', '烫手', '有雾', '没避障', '遮挡', '歪', '撞机', '断裂', '不准', '偏移', '漂干扰', '变形', '回不来', '中断', '噪音', '生锈', '抖动', '过载', '飘', '乱飞', '异响', '坏点', '短路', 'bug', '裂开', '侧飞', '鼓包', '磨损', '起雾', '俯冲', '停不下来', '下降', '晃动', '坠机', '烧', '停转', '错误', '迫降', '黑屏', '误报', '发热', '避障关', '弱', '倾斜', '降到', '关机', '撞击', '模糊', '进水', '关避障', '崩溃', '拉胯', '抗风差', '太差', '关不了', '断电', '报警', '失控', '缺陷', '落地', '划痕断联', '掉下来', '损坏', '故障', '降低', '没电', '丢失', '瑕疵', '停桨', '接触不良', '断开连接', '失效', '异常', '坠毁', '关闭', '出错', '脱落', '放电', '摔坏', '破损',

In [41]:
# 1
# 函数：计算并返回与给定词相似度最高的N个词
def top_similar_words(target_word, vocab_set, word_vectors, top_n=10):
    if target_word in word_vectors:
        # 获取与目标词最相似的词汇
        similar_words = word_vectors.most_similar(target_word, topn=top_n)
        # 筛选出在 vocab_set 中的相似词，并按相似度降序排列
        filtered_similar_words = [(sim_word, sim_score) for sim_word, sim_score in similar_words if sim_word in vocab_set]
        # 返回前 top_n 个最相似的词及其相似度
        return sorted(filtered_similar_words, key=lambda x: x[1], reverse=True)[:top_n]
    else:
        return []

# 初始化空列表来收集所有相似词数据
similar_words_Dc = []
similar_words_Df = []

# 计算Dc中每个词与Dn中词的相似度，并收集数据
for dc_word in Dc:
    similar_words = top_similar_words(dc_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Dc.append([dc_word, dn_word, similarity])

# 计算Df中每个词与Dn中词的相似度，并收集数据
for df_word in Df:
    similar_words = top_similar_words(df_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Df.append([df_word, dn_word, similarity])

# 创建DataFrame
similar_words_dc = pd.DataFrame(similar_words_Dc, columns=['Target Word', 'Similar Word', 'Similarity'])
similar_words_df = pd.DataFrame(similar_words_Df, columns=['Target Word', 'Similar Word', 'Similarity'])

similar_words_dc.to_csv(r'E:\UAV\中文停用词表分词结果\组件相似词2.csv', index=False, encoding='utf_8_sig')
similar_words_df.to_csv(r'E:\UAV\中文停用词表分词结果\故障相似词2.csv', index=False, encoding='utf_8_sig')

print(similar_words_dc)
print(similar_words_df)

    Target Word Similar Word  Similarity
0            相机           露营    0.861989
1            相机         Vlog    0.856977
2            相机         4k60    0.856511
3            相机         dlog    0.853574
4            相机          分辨率    0.852882
..          ...          ...         ...
625          接口           背囊    0.932634
626          接口            插    0.931866
627          接口            灯    0.930912
628          接口           剩余    0.929784
629          接口            闪    0.929568

[630 rows x 3 columns]
    Target Word Similar Word  Similarity
0            警告           地标    0.982826
1            警告           楼房    0.979769
2            警告           往回    0.978548
3            警告          十几米    0.978407
4            警告           房子    0.978347
..          ...          ...         ...
759           烫           改装    0.951506
760           烫           报废    0.950921
761           烫           通病    0.950717
762           烫           一格    0.949187
763           烫           针脚    0

In [43]:
# 2
# 相似度计算结果中挑选出来的要加入Dc和Df的词
selected_for_Dc = set(["橡胶圈", "散热器", "机架", "程序", "桨", "支架", "控系统", "无线电", "红外", "机架", "架", "APP"])  
selected_for_Df = set(["充不进", "偏", "断", "电流", "高温", "掉下去", "抽风", "松", "通病", "报废", "自由落体", "错乱", "飘走", "磕", "顿感",
                       "松", "坏掉", "压坏", "掉电", "废", "烂", "衰减", "裂", "失灵", "警报", "闪烁", "闪", "滋滋", "变色", "报废", "松"
                       "脆弱"])  

# 将选中的词添加到对应的词典中
Dc = Dc.union(selected_for_Dc)
Df = Df.union(selected_for_Df)

# 打印更新后的词典
print("更新后的Dc:", Dc)
print("更新后的Df:", Df)

# 创建噪声词词典Dn，包含所有不在Dc和Df中的词
# 注意：这里假设S_flat包含了所有分词后的数据集词汇
Dn = set([word for sentence in S for word in sentence]) - Dc - Df

print("噪声词典Dn:", Dn)

更新后的Dc: {'相机', '稳定器', '螺丝柱', '云台', 'APP', '圈圈', '机体', '桨', '轴承', '橡胶圈', '电源', '遥控', '罩', '硬件', '架', '桨叶', '传感器', '显示器', 'o4', '充电器', '机身', 'o3', '镜头', '机翼', 'CMOS', '外壳', '螺旋桨', '机架', '导航', '雷达', '电机', '信号', '壳', '光圈', '核心', '红外', '轴臂', '哈苏', '散热', '软件', '系统', '模组', '机臂', '指示灯', '控系统', '电芯', '散热器', '芯片', '程序', 'usb', '叶片', '4g', '排线', '遥控器', 'rc2', '模块', '飞控', '图传', '电线', '按钮', '卡扣', '整机', '摇杆', '无线电', '主摄', '摄像头', '算法', '避障', '电池', '风扇', '主板', '手柄', 'gps', '内存卡', '天线', '固件', '旋翼', '充电', '三轴', '屏幕', '激光雷达', 'rc', '支架', '接口'}
更新后的Df: {'磕', '砸撞坏', '警告', '爆炸', '翻滚', '掉落', '高温', '坠落', '松动', '报废', '松', '烫手', '闪', '有雾', '变色', '没避障', '遮挡', '歪', '撞机', '断裂', '失灵', '不准', '偏移', '漂干扰', '变形', '回不来', '中断', '噪音', '烂', '裂', '生锈', '抖动', '错乱', '过载', '飘', '乱飞', '异响', '坏点', '压坏', '抽风', '飘走', '短路', 'bug', '断', '裂开', '侧飞', '鼓包', '磨损', '起雾', '充不进', '电流', '俯冲', '停不下来', '下降', '晃动', '坠机', '烧', '停转', '错误', '松脆弱', '迫降', '黑屏', '误报', '发热', '掉电', '自由落体', '避障关', '弱', '倾斜', '降到', '关机', '撞击', '模糊', '进水', '关避障', '崩溃', '

In [45]:
# 2
# 函数：计算并返回与给定词相似度最高的N个词
def top_similar_words(target_word, vocab_set, word_vectors, top_n=10):
    if target_word in word_vectors:
        # 获取与目标词最相似的词汇
        similar_words = word_vectors.most_similar(target_word, topn=top_n)
        # 筛选出在 vocab_set 中的相似词，并按相似度降序排列
        filtered_similar_words = [(sim_word, sim_score) for sim_word, sim_score in similar_words if sim_word in vocab_set]
        # 返回前 top_n 个最相似的词及其相似度
        return sorted(filtered_similar_words, key=lambda x: x[1], reverse=True)[:top_n]
    else:
        return []

# 初始化空列表来收集所有相似词数据
similar_words_Dc = []
similar_words_Df = []

# 计算Dc中每个词与Dn中词的相似度，并收集数据
for dc_word in Dc:
    similar_words = top_similar_words(dc_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Dc.append([dc_word, dn_word, similarity])

# 计算Df中每个词与Dn中词的相似度，并收集数据
for df_word in Df:
    similar_words = top_similar_words(df_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Df.append([df_word, dn_word, similarity])

# 创建DataFrame
similar_words_dc = pd.DataFrame(similar_words_Dc, columns=['Target Word', 'Similar Word', 'Similarity'])
similar_words_df = pd.DataFrame(similar_words_Df, columns=['Target Word', 'Similar Word', 'Similarity'])

similar_words_dc.to_csv(r'E:\UAV\中文停用词表分词结果\组件相似词3.csv', index=False, encoding='utf_8_sig')
similar_words_df.to_csv(r'E:\UAV\中文停用词表分词结果\故障相似词3.csv', index=False, encoding='utf_8_sig')

print(similar_words_dc)
print(similar_words_df)

    Target Word Similar Word  Similarity
0            相机           露营    0.861989
1            相机         Vlog    0.856977
2            相机         4k60    0.856511
3            相机         dlog    0.853574
4            相机          分辨率    0.852882
..          ...          ...         ...
703          支架            扔    0.958491
704          接口           背囊    0.932634
705          接口            插    0.931866
706          接口            灯    0.930912
707          接口           剩余    0.929784

[708 rows x 3 columns]
    Target Word Similar Word  Similarity
0             磕           空调    0.989076
1             磕           浆叶    0.982218
2             磕           满载    0.980420
3             磕          急刹车    0.980346
4             磕           缓冲    0.980045
..          ...          ...         ...
967           烫           飞完    0.954094
968           烫           换上    0.952470
969           烫           改装    0.951506
970           烫           一格    0.949187
971           烫           针脚    0

In [47]:
# 3
# 相似度计算结果中挑选出来的要加入Dc和Df的词
selected_for_Dc = set(["机头", "感应器", "控制系统", "动力系统", "飞机", "硬件"])  
selected_for_Df = set(["停机", "摇晃", "脱焊", "刮", "发烫", "无法控制", "漏洞", "断连", "压差", "弄坏", "过低", "损伤", "自燃", "不够用", 
                       "没了", ])  

# 将选中的词添加到对应的词典中
Dc = Dc.union(selected_for_Dc)
Df = Df.union(selected_for_Df)

# 打印更新后的词典
print("更新后的Dc:", Dc)
print("更新后的Df:", Df)

# 创建噪声词词典Dn，包含所有不在Dc和Df中的词
# 注意：这里假设S_flat包含了所有分词后的数据集词汇
Dn = set([word for sentence in S for word in sentence]) - Dc - Df

print("噪声词典Dn:", Dn)

更新后的Dc: {'相机', '稳定器', '螺丝柱', '云台', 'APP', '圈圈', '机体', '桨', '轴承', '橡胶圈', '电源', '遥控', '罩', '控制系统', '硬件', '架', '桨叶', '传感器', '显示器', 'o4', '充电器', '机身', 'o3', '镜头', '机翼', 'CMOS', '外壳', '螺旋桨', '机架', '飞机', '导航', '雷达', '电机', '信号', '壳', '感应器', '光圈', '核心', '红外', '轴臂', '哈苏', '散热', '软件', '系统', '模组', '机臂', '指示灯', '控系统', '电芯', '散热器', '芯片', '程序', 'usb', '叶片', '4g', '排线', '遥控器', 'rc2', '模块', '机头', '飞控', '图传', '电线', '按钮', '卡扣', '整机', '摇杆', '无线电', '主摄', '摄像头', '算法', '避障', '电池', '风扇', '主板', '手柄', 'gps', '内存卡', '天线', '固件', '旋翼', '充电', '三轴', '屏幕', '激光雷达', 'rc', '动力系统', '支架', '接口'}
更新后的Df: {'磕', '砸撞坏', '警告', '漏洞', '爆炸', '翻滚', '掉落', '刮', '高温', '坠落', '松动', '报废', '松', '烫手', '闪', '有雾', '变色', '没避障', '遮挡', '歪', '撞机', '断裂', '失灵', '不准', '偏移', '漂干扰', '变形', '回不来', '摇晃', '中断', '噪音', '烂', '裂', '生锈', '过低', '抖动', '错乱', '过载', '飘', '乱飞', '异响', '弄坏', '坏点', '压坏', '抽风', '飘走', '短路', 'bug', '断', '发烫', '自燃', '裂开', '侧飞', '断连', '鼓包', '磨损', '起雾', '充不进', '电流', '俯冲', '停不下来', '下降', '晃动', '坠机', '烧', '停转', '错误', '松脆弱', '脱焊', '迫降', '黑屏', 

In [49]:
# 3
# 函数：计算并返回与给定词相似度最高的N个词
def top_similar_words(target_word, vocab_set, word_vectors, top_n=10):
    if target_word in word_vectors:
        # 获取与目标词最相似的词汇
        similar_words = word_vectors.most_similar(target_word, topn=top_n)
        # 筛选出在 vocab_set 中的相似词，并按相似度降序排列
        filtered_similar_words = [(sim_word, sim_score) for sim_word, sim_score in similar_words if sim_word in vocab_set]
        # 返回前 top_n 个最相似的词及其相似度
        return sorted(filtered_similar_words, key=lambda x: x[1], reverse=True)[:top_n]
    else:
        return []

# 初始化空列表来收集所有相似词数据
similar_words_Dc = []
similar_words_Df = []

# 计算Dc中每个词与Dn中词的相似度，并收集数据
for dc_word in Dc:
    similar_words = top_similar_words(dc_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Dc.append([dc_word, dn_word, similarity])

# 计算Df中每个词与Dn中词的相似度，并收集数据
for df_word in Df:
    similar_words = top_similar_words(df_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Df.append([df_word, dn_word, similarity])

# 创建DataFrame
similar_words_dc = pd.DataFrame(similar_words_Dc, columns=['Target Word', 'Similar Word', 'Similarity'])
similar_words_df = pd.DataFrame(similar_words_Df, columns=['Target Word', 'Similar Word', 'Similarity'])

similar_words_dc.to_csv(r'E:\UAV\中文停用词表分词结果\组件相似词4.csv', index=False, encoding='utf_8_sig')
similar_words_df.to_csv(r'E:\UAV\中文停用词表分词结果\故障相似词4.csv', index=False, encoding='utf_8_sig')

print(similar_words_dc)
print(similar_words_df)

    Target Word Similar Word  Similarity
0            相机           露营    0.861989
1            相机         Vlog    0.856977
2            相机         4k60    0.856511
3            相机         dlog    0.853574
4            相机          分辨率    0.852882
..          ...          ...         ...
743          支架            扔    0.958491
744          接口           背囊    0.932634
745          接口            插    0.931866
746          接口            灯    0.930912
747          接口           剩余    0.929784

[748 rows x 3 columns]
     Target Word Similar Word  Similarity
0              磕           空调    0.989076
1              磕           浆叶    0.982218
2              磕           满载    0.980420
3              磕          急刹车    0.980346
4              磕           缓冲    0.980045
...          ...          ...         ...
1062           烫           飞完    0.954094
1063           烫           换上    0.952470
1064           烫           改装    0.951506
1065           烫           一格    0.949187
1066           烫      

In [51]:
# 3
# 相似度计算结果中挑选出来的要加入Dc和Df的词
selected_for_Dc = set(["连接线", "电路板", "针脚", "内部", "轴", "臂章", "陀螺仪", "广角镜头", "定位系统", "通讯"])  
selected_for_Df = set(["连不上", "不避障", "漂移", "响", "摔下来", "划伤", "抽搐", "骤降", "避不开", "太吵", "蹭", "不好","卡"])  

# 将选中的词添加到对应的词典中
Dc = Dc.union(selected_for_Dc)
Df = Df.union(selected_for_Df)

# 打印更新后的词典
print("更新后的Dc:", Dc)
print("更新后的Df:", Df)

# 创建噪声词词典Dn，包含所有不在Dc和Df中的词
# 注意：这里假设S_flat包含了所有分词后的数据集词汇
Dn = set([word for sentence in S for word in sentence]) - Dc - Df

print("噪声词典Dn:", Dn)

更新后的Dc: {'相机', '稳定器', '螺丝柱', '云台', 'APP', '圈圈', '机体', '桨', '内部', '轴承', '橡胶圈', '电源', '遥控', '罩', '控制系统', '硬件', '架', '桨叶', '传感器', '显示器', 'o4', '连接线', '充电器', '机身', '陀螺仪', 'o3', '镜头', '机翼', 'CMOS', '外壳', '定位系统', '螺旋桨', '机架', '飞机', '导航', '雷达', '电机', '信号', '壳', '感应器', '光圈', '核心', '红外', '广角镜头', '轴臂', '哈苏', '散热', '软件', '系统', '模组', '机臂', '指示灯', '控系统', '电芯', '散热器', '芯片', '程序', '通讯', 'usb', '叶片', '4g', '排线', '电路板', '遥控器', 'rc2', '模块', '臂章', '机头', '飞控', '图传', '电线', '按钮', '卡扣', '针脚', '整机', '摇杆', '轴', '无线电', '主摄', '摄像头', '算法', '避障', '电池', '风扇', '主板', '手柄', 'gps', '内存卡', '天线', '固件', '旋翼', '充电', '三轴', '屏幕', '激光雷达', 'rc', '动力系统', '支架', '接口'}
更新后的Df: {'磕', '砸撞坏', '警告', '漏洞', '爆炸', '高温', '坠落', '报废', '变色', '没避障', '响', '偏移', '变形', '回不来', '中断', '烂', '过载', '飘', '乱飞', '弄坏', '压坏', '抽风', '骤降', '飘走', '发烫', '自燃', '裂开', '侧飞', '磨损', '起雾', '电流', '俯冲', '停不下来', '晃动', '坠机', '停转', '错误', '脱焊', '迫降', '自由落体', '弱', '没了', '降到', '关机', '模糊', '关避障', '崩溃', '衰减', '顿感', '警报', '不避障', '拉胯', '太差', '关不了', '报警', '划痕断联', '滋滋', '掉下来', '损坏

In [53]:
# 3
# 函数：计算并返回与给定词相似度最高的N个词
def top_similar_words(target_word, vocab_set, word_vectors, top_n=10):
    if target_word in word_vectors:
        # 获取与目标词最相似的词汇
        similar_words = word_vectors.most_similar(target_word, topn=top_n)
        # 筛选出在 vocab_set 中的相似词，并按相似度降序排列
        filtered_similar_words = [(sim_word, sim_score) for sim_word, sim_score in similar_words if sim_word in vocab_set]
        # 返回前 top_n 个最相似的词及其相似度
        return sorted(filtered_similar_words, key=lambda x: x[1], reverse=True)[:top_n]
    else:
        return []

# 初始化空列表来收集所有相似词数据
similar_words_Dc = []
similar_words_Df = []

# 计算Dc中每个词与Dn中词的相似度，并收集数据
for dc_word in Dc:
    similar_words = top_similar_words(dc_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Dc.append([dc_word, dn_word, similarity])

# 计算Df中每个词与Dn中词的相似度，并收集数据
for df_word in Df:
    similar_words = top_similar_words(df_word, Dn, word_vectors, top_n=10)
    for dn_word, similarity in similar_words:
        similar_words_Df.append([df_word, dn_word, similarity])

# 创建DataFrame
similar_words_dc = pd.DataFrame(similar_words_Dc, columns=['Target Word', 'Similar Word', 'Similarity'])
similar_words_df = pd.DataFrame(similar_words_Df, columns=['Target Word', 'Similar Word', 'Similarity'])

similar_words_dc.to_csv(r'E:\UAV\中文停用词表分词结果\组件相似词5.csv', index=False, encoding='utf_8_sig')
similar_words_df.to_csv(r'E:\UAV\中文停用词表分词结果\故障相似词5.csv', index=False, encoding='utf_8_sig')

print(similar_words_dc)
print(similar_words_df)

    Target Word Similar Word  Similarity
0            相机           露营    0.861989
1            相机         Vlog    0.856977
2            相机         4k60    0.856511
3            相机         dlog    0.853574
4            相机          分辨率    0.852882
..          ...          ...         ...
833          支架            扔    0.958491
834          接口           背囊    0.932634
835          接口            插    0.931866
836          接口            灯    0.930912
837          接口           剩余    0.929784

[838 rows x 3 columns]
     Target Word Similar Word  Similarity
0              磕           空调    0.989076
1              磕           浆叶    0.982218
2              磕           满载    0.980420
3              磕          急刹车    0.980346
4              磕           缓冲    0.980045
...          ...          ...         ...
1164          失联           断联    0.906165
1165          报错           现象    0.966781
1166          报错            v    0.949057
1167          报错            拧    0.948319
1168          报错      

In [55]:
# 4
# 相似度计算结果中挑选出来的要加入Dc和Df的词
selected_for_Dc = set(["马达"])  
selected_for_Df = set([""])  

# 将选中的词添加到对应的词典中
Dc = Dc.union(selected_for_Dc)
Df = Df.union(selected_for_Df)

# 打印更新后的词典
print("更新后的Dc:", Dc)
print("更新后的Df:", Df)

# 创建噪声词词典Dn，包含所有不在Dc和Df中的词
# 注意：这里假设S_flat包含了所有分词后的数据集词汇
Dn = set([word for sentence in S for word in sentence]) - Dc - Df

print("噪声词典Dn:", Dn)

更新后的Dc: {'相机', '稳定器', '螺丝柱', '云台', 'APP', '圈圈', '机体', '桨', '内部', '轴承', '橡胶圈', '电源', '遥控', '罩', '控制系统', '硬件', '架', '桨叶', '传感器', '显示器', 'o4', '连接线', '充电器', '机身', '陀螺仪', 'o3', '镜头', '机翼', 'CMOS', '外壳', '定位系统', '螺旋桨', '机架', '飞机', '导航', '雷达', '电机', '信号', '壳', '感应器', '光圈', '核心', '红外', '广角镜头', '轴臂', '哈苏', '散热', '软件', '系统', '模组', '机臂', '指示灯', '控系统', '电芯', '散热器', '芯片', '程序', '通讯', 'usb', '叶片', '4g', '排线', '电路板', '遥控器', 'rc2', '模块', '臂章', '机头', '飞控', '图传', '电线', '按钮', '卡扣', '针脚', '整机', '摇杆', '轴', '无线电', '主摄', '摄像头', '算法', '避障', '电池', '风扇', '主板', '手柄', 'gps', '内存卡', '天线', '固件', '马达', '旋翼', '充电', '三轴', '屏幕', '激光雷达', 'rc', '动力系统', '支架', '接口'}
更新后的Df: {'磕', '', '砸撞坏', '警告', '漏洞', '爆炸', '高温', '坠落', '报废', '变色', '没避障', '响', '偏移', '变形', '回不来', '中断', '烂', '过载', '飘', '乱飞', '弄坏', '压坏', '抽风', '骤降', '飘走', '发烫', '自燃', '裂开', '侧飞', '磨损', '起雾', '电流', '俯冲', '停不下来', '晃动', '坠机', '停转', '错误', '脱焊', '迫降', '自由落体', '弱', '没了', '降到', '关机', '模糊', '关避障', '崩溃', '衰减', '顿感', '警报', '不避障', '拉胯', '太差', '关不了', '报警', '划痕断联', '滋滋', 